# Confidence Is Not Probability

Two sentences about the same 95% interval: *"intervals constructed this way contain the true parameter 95% of the time"* — correct. *"There is a 95% probability that the population parameter falls within this interval"* — **wrong**. The difference is subtle enough that the course pauses a whole video on it: $\mu$ is **fixed but unknown** — it has no probability distribution, so for any given interval it simply *is* inside or *isn't*. What's random is the interval (its center is a sample mean), and the 95% is the **success rate of the construction procedure** over repeated sampling — never a property of the one interval in your hands. It is based on the seventh video of Lesson 7 (Confidence Intervals).

## 1. Introduction

Notebook 01's slogan was "the key never moves." This notebook turns that slogan into a rule about *language*: which probability statements are legitimate, which are illegal, and why — sharpening the single most common misinterpretation in all of statistics.

After completing this notebook you will be able to:

- State the correct and the incorrect reading of "95% confidence" and pinpoint where they diverge.
- Explain why $\mu$ admits no probability statements: fixed and unknown is not the same as random.
- Distinguish before-sampling statements (about the random $\bar{X}$ — legitimate) from after-sampling ones (about a realized interval — degenerate).
- Demonstrate by simulation that each interval's containment is a plain 0 or 1, whose *long-run average* is the confidence level.
- Connect the restriction to Lesson 6's frequentist–Bayesian divide, and defuse a second subtle trap about future samples.

## 2. Theory

### Two sentences, one word apart

You compute $\bar{x}$ and a 95% confidence interval around it.

- ✅ *"Confidence intervals constructed like this contain the true population parameter 95% of the time."*
- ❌ *"There is a 95% probability that the population parameter falls within this confidence interval."*

### Why the second is wrong: $\mu$ is not random

A defining characteristic of the population mean $\mu$: it is **fixed but unknown**. Unknown is not random — $\mu$ **does not have a probability distribution**; it is always the same value for a given population. So for a given interval, $\mu$ *is in it or it is not*. It will not "fall within the interval 95% of the time": there is no falling, no repetition, no randomness on $\mu$'s side at all. The probability that this specific interval contains $\mu$ is either 1 or 0 — you just don't know which.

### What the 95% actually describes

The **sample mean** does have a probability distribution — the sampling distribution — and the interval is tied to it: a different sample, a different $\bar{x}$, a different interval. "95% confident" is a statement about **repeating the sampling experiment**: take many samples, build the interval each time, and 95% of those intervals will contain $\mu$. The confidence level is the **success rate of the interval-construction procedure** — a grade for the recipe, not for any single dish it produced.

### Before and after

The clean way to keep it straight is a timeline:

- **Before sampling**, $\bar{X}$ is random, so the *interval-to-be* is random, and $P(\bar{X} - m \le \mu \le \bar{X} + m) = 0.95$ is a perfectly legal statement — it's a probability about $\bar{X}$.
- **After sampling**, $\bar{x} = 168.3$ is a number, the interval $[161.3, 175.3]$ is a pair of numbers, and $\mu$ was always a number. Nothing random remains; the only honest residue of the 95% is "this interval came from a recipe that succeeds 95% of the time."

The coin analogy: flip a fair coin and cover it with your hand. *Before* the flip, "heads with probability 0.5" describes the flip. *After*, the coin under your hand is already heads or tails — the 0.5 now describes your ignorance of a settled fact, not any remaining randomness of the coin. A computed confidence interval is a covered coin.

### A note for Lesson 6 veterans

The restriction is a feature of the **frequentist** framework, where parameters are fixed and only data is random. A **Bayesian** — who models $\mu$ itself as a random variable with a prior — is allowed to say "given the data, $\mu$ lies in this range with 95% probability" (a *credible* interval, built from the posterior). Same data, different model of what "probability" attaches to. Within this lesson's frequentist machinery, sentence two stays illegal.

## 3. Worked Examples

### Example 1: a language audit

| statement | verdict |
|---|---|
| "95% of intervals built by this procedure contain $\mu$." | ✅ the definition |
| "There's a 95% chance $\mu$ is between 161.3 and 175.3." | ❌ $\mu$ is fixed; it's in or it's out |
| "Before I sample, the probability my interval will capture $\mu$ is 95%." | ✅ a statement about random $\bar{X}$ |
| "$\mu$ falls in my interval 95% of the time." | ❌ $\mu$ never falls anywhere; intervals move, $\mu$ doesn't |
| "I'm 95% confident $\mu$ is between 161.3 and 175.3." | ✅ *if* "confident" is understood as recipe-success-rate — that's precisely what the word is for |

### Example 2: the degenerate probability

Suppose (omnisciently) $\mu = 170$ and your realized interval is $[161.3, 175.3]$. Then "P($\mu$ in the interval)" $= 1$ — today, tomorrow, always. Had your sample been unluckier — interval $[155.0, 169.0]$ — it would be $0$. No repetition of the *question* changes either answer; only re-running the *sampling* produces new intervals, and that is where the 95% lives.

### Example 3: why the distinction has teeth

If sentence two were true, you could bet at 19:1 odds on any single computed interval. But a stubborn adversary who knows $\mu$ would simply accept your bets only on the ~5% of intervals that missed — and win every single one. Probability statements about fixed facts aren't harmless shorthand; they misprice real decisions. What protects you across *many* bets is exactly the recipe's long-run 95% — which is all the confidence level ever promised.

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(13)

MU_TRUE, SIGMA, N = 170.0, 25.0, 49
MARGIN = float(sp.sqrt(2) * sp.erfinv(sp.Float(0.95))) * SIGMA / np.sqrt(N)

### One interval: containment is 0 or 1, and asking again doesn't help

Compute one interval, then "ask nature" repeatedly whether $\mu$ is inside. The answer never changes — there is no 95% anywhere in sight.

In [ ]:
x_bar = rng.normal(MU_TRUE, SIGMA / np.sqrt(N))
interval = (x_bar - MARGIN, x_bar + MARGIN)
print(f"the one interval you actually get: [{interval[0]:.1f}, {interval[1]:.1f}]")

answers = [interval[0] <= MU_TRUE <= interval[1] for _ in range(10)]
print(f"asking 'is mu inside?' ten times: {answers}")
print("Same answer every time -- for a realized interval, containment is a fixed")
print("fact (here we can peek because we simulated mu; in real life you never can).")

### Many intervals: the 95% appears exactly where the repetition is

Rerun the *sampling* 2,000 times. Each interval's containment is still a hard 0 or 1 — but the running average of those 0s and 1s converges to 0.95. Confidence is a long-run success rate of the procedure.

In [ ]:
trials = 2_000
x_bars = rng.normal(MU_TRUE, SIGMA / np.sqrt(N), size=trials)
contained = (np.abs(x_bars - MU_TRUE) <= MARGIN).astype(int)
running_rate = np.cumsum(contained) / np.arange(1, trials + 1)

print(f"first 20 containment outcomes: {contained[:20]}")
print(f"any value other than 0 or 1? {set(contained) - {0, 1} or 'no'}")
print(f"success rate after {trials} repetitions: {running_rate[-1]:.1%}")

plt.figure(figsize=(9.5, 4.4))
plt.plot(running_rate, color="tab:blue", linewidth=1.5,
         label="running success rate of the recipe")
plt.axhline(0.95, color="black", linestyle=":", label="confidence level 0.95")
plt.ylim(0.85, 1.0)
plt.xlabel("number of repeated sampling experiments")
plt.ylabel("fraction of intervals containing mu")
plt.title("Each outcome is 0 or 1; only their long-run average is 95%")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 5. Visualizations

### What is random, and what is not

Left: the world the 95% lives in — intervals jumping around a motionless $\mu$ as sampling repeats. Right: the world you inhabit after sampling — one frozen interval, one fixed $\mu$, and a question whose answer is already settled (shown for both a lucky and an unlucky sample).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.2), sharex=True)

# left: repetition -- 40 intervals
means_40 = rng.normal(MU_TRUE, SIGMA / np.sqrt(N), size=40)
for i, m in enumerate(means_40):
    ok = abs(m - MU_TRUE) <= MARGIN
    axes[0].plot([m - MARGIN, m + MARGIN], [i, i],
                 color="tab:green" if ok else "tab:red", linewidth=1.8, alpha=0.85)
axes[0].axvline(MU_TRUE, color="black", linestyle="--", linewidth=1.5)
axes[0].set_title("BEFORE / in repetition: intervals are random,\n95% of them capture the fixed mu", fontsize=10)
axes[0].set_ylabel("repetition of the experiment")
axes[0].set_xlabel("height (cm)")

# right: two frozen intervals
lucky, unlucky = 168.3, 160.0
for y, (center, label) in enumerate([(lucky, "P(mu inside) = 1"),
                                     (unlucky, "P(mu inside) = 0")]):
    ok = abs(center - MU_TRUE) <= MARGIN
    axes[1].plot([center - MARGIN, center + MARGIN], [y * 6 + 15, y * 6 + 15],
                 color="tab:green" if ok else "tab:red", linewidth=6,
                 solid_capstyle="round")
    axes[1].annotate(f"a realized interval: {label}",
                     xy=(center, y * 6 + 17), ha="center", fontsize=10)
axes[1].axvline(MU_TRUE, color="black", linestyle="--", linewidth=1.5)
axes[1].annotate(r"$\mu$ (fixed)", xy=(MU_TRUE + 0.5, 36), fontsize=10)
axes[1].set_ylim(8, 40)
axes[1].set_yticks([])
axes[1].set_title("AFTER: one interval, zero randomness --\ncontainment is a settled 0 or 1", fontsize=10)
axes[1].set_xlabel("height (cm)")

for ax in axes:
    ax.grid(True, alpha=0.3, axis="x")
plt.suptitle("The 95% belongs to the left panel; you live in the right one")
plt.tight_layout()
plt.show()

### The whole story on one page

Everything above in a single sheet: one fixed interval vs. many repeated ones, a language audit of right and wrong statements, and the long-run 95%.

<img src="img/06_confidence_vs_probability.png" alt="Concept sheet for this notebook" width="900">

## 6. Exercises

### Basic

**Exercise 1.** Verdict, with a one-line reason each: (a) "the population proportion has a 95% chance of exceeding 0.66"; (b) "95% of samples give intervals that catch the true proportion"; (c) "before polling, there's a 95% probability our interval will catch it."

**Exercise 2.** Explain the covered-coin analogy in your own words: what plays the flip, what plays the covered coin, and where does the 0.5 (resp. 0.95) legitimately live?

### Intermediate

**Exercise 3.** Repeat the running-success-rate simulation at the 80% level and confirm both facts at once: every individual outcome is 0/1, and the average converges to 0.80.

**Exercise 4.** A news article writes: *"There is a 95% probability that the true unemployment rate lies between 3.8% and 4.4%."* Rewrite the sentence so it is statistically correct without losing its informativeness, and note which single concept the fix hinges on.

### Advanced

**Exercise 5.** *The Bayesian door.* Lesson 6 built posteriors for exactly this situation. Give $\mu$ the prior $N(165, 10^2)$, observe $\bar{x} = 168.3$ from $n = 49$ (with $\sigma = 25$ known, posterior variance $= (1/\sigma_0^2 + n/\sigma^2)^{-1}$), and compute the genuine posterior probability that $\mu$ lies in the frequentist interval $[161.3, 175.3]$. Why is this sentence legal for the Bayesian when it was illegal all notebook long?

**Exercise 6.** *A second trap.* Tempting claim: "95% of *future sample means* will land inside my computed interval $\bar{x} \pm 1.96\,\sigma/\sqrt{n}$." Simulate it: for many observed $\bar{x}$'s, measure what fraction of fresh sample means land in each one's interval. Show the average capture rate is about 83%, not 95%, and explain the missing 12 points (hint: *two* random sample means are in play, and the variance of a difference adds).

## 7. Solutions

In [ ]:
# Exercise 1: verdicts
print("(a) WRONG -- the population proportion is fixed; it has no chances.")
print("(b) RIGHT -- a statement about the procedure across samples.")
print("(c) RIGHT -- before sampling, the interval is random, so the probability")
print("    statement attaches to it (not to the parameter).")

In [ ]:
# Exercise 2: the covered coin
print("The flip = drawing the sample (genuinely random; probabilities legal).")
print("The covered coin = the computed interval next to the fixed mu: the outcome")
print("is already settled, merely unknown to you. The 0.5 / 0.95 lives in the")
print("procedure -- flips in general, interval-recipes in general -- never in the")
print("settled-but-hidden particular case.")

In [ ]:
# Exercise 3: running rate at 80%
margin_80 = float(sp.sqrt(2) * sp.erfinv(sp.Float(0.80))) * SIGMA / np.sqrt(N)
xb3 = rng.normal(MU_TRUE, SIGMA / np.sqrt(N), size=2_000)
c3 = (np.abs(xb3 - MU_TRUE) <= margin_80).astype(int)
print(f"outcomes observed: {sorted(set(c3))} (nothing between 0 and 1)")
print(f"running rate after 2000 repetitions: {c3.mean():.1%} (target 80%)")

In [ ]:
# Exercise 4: fixing the journalist
print('Correct version: "The unemployment rate was estimated at 4.1% with a 95%')
print('confidence interval of 3.8% to 4.4% -- a range produced by a method that')
print('captures the true rate in 95% of surveys."')
print("The fix hinges on relocating the 95% from the fixed true rate to the")
print("method (the interval-generating procedure).")

In [ ]:
# Exercise 5: the Bayesian door
sigma0, mu0 = 10.0, 165.0                     # prior N(165, 10^2)
x_obs, n5 = 168.3, 49
post_var = 1 / (1 / sigma0**2 + n5 / SIGMA**2)
post_mean = post_var * (mu0 / sigma0**2 + n5 * x_obs / SIGMA**2)
post_sd = np.sqrt(post_var)

lo5, hi5 = 161.3, 175.3
Phi = lambda z: 0.5 * (1 + float(sp.erf(sp.Float(z) / sp.sqrt(2))))
prob_in = Phi((hi5 - post_mean) / post_sd) - Phi((lo5 - post_mean) / post_sd)
print(f"posterior: N({post_mean:.2f}, {post_sd:.2f}^2)")
print(f"P(mu in [161.3, 175.3] | data) = {prob_in:.1%}  -- a legal sentence now!")
print("Legal because the Bayesian MODELS mu as random (the prior); probability")
print("then measures belief about mu, updated by data -- Lesson 6's machinery.")
print("The frequentist mu has no distribution, so the same sentence has no meaning.")

In [ ]:
# Exercise 6: the future-sample-mean trap
se = SIGMA / np.sqrt(N)
observed = rng.normal(MU_TRUE, se, size=2_000)
capture_rates = []
for xb in observed[:500]:
    future = rng.normal(MU_TRUE, se, size=400)
    capture_rates.append(np.mean(np.abs(future - xb) <= MARGIN))
avg_capture = np.mean(capture_rates)

exact = float(sp.erf(sp.Float(1.959963985) / sp.sqrt(2) / sp.sqrt(2)))
print(f"average fraction of future sample means inside an observed interval: "
      f"{avg_capture:.1%}")
print(f"theory: P(|Xbar1 - Xbar2| <= 1.96 se) = P(|Z| <= 1.96/sqrt(2)) = {exact:.1%}")
print("\nThe interval was sized for ONE random sample mean around fixed mu. A")
print("future comparison has TWO random sample means, and the variance of their")
print("difference doubles -- the effective z is 1.96/sqrt(2) = 1.39, worth ~83%.")

## 8. Conclusion

- **Correct:** "intervals constructed this way contain $\mu$ 95% of the time." **Wrong:** "$\mu$ has a 95% probability of being in this interval." One sentence grades the recipe; the other pretends a fixed number is random.
- $\mu$ is **fixed but unknown** — it has no probability distribution, so a realized interval contains it with probability 1 or 0, never 0.95. All the randomness (and hence all the probability) belongs to $\bar{X}$ and the interval built on it.
- Before sampling, probability statements about the interval-to-be are legal; after sampling, only the recipe's long-run success rate survives — as the simulation shows, individual outcomes are pure 0s and 1s whose average is 95%.
- Two doors out of the restriction, both principled: repeat the experiment (frequentist — where the 95% lives), or put a prior on $\mu$ (Bayesian — Lesson 6's credible intervals, where the forbidden sentence becomes meaningful).
- Bonus trap defused: a 95% interval captures only ~83% of *future sample means* — it was never sized for two-sided randomness.

Subtle, but now clarified — next, the last standing assumption: what happens when $\sigma$ is unknown.